# Qwen2-VL Downstream Evaluation
This notebook runs exact match (EM), F1, Latency and FLOPS evaluations for different retrieval methods.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/NguyenKhanh2603/Uncertainty-Aware-Iterative-RAG.git"
CODE_COMMIT = "a9a80508f4449486a64199366ecb1354e01d0ff0"
PROJECT_DIR = Path("/content/uncertainty-aware-rag")

uv_executable = shutil.which("uv")
if uv_executable is None:
    installer = Path("/tmp/install-uv.sh")
    subprocess.check_call(["curl", "-LsSf", "https://astral.sh/uv/install.sh", "-o", str(installer)])
    subprocess.check_call(["sh", str(installer)])
    uv_executable = str(Path.home() / ".local" / "bin" / "uv")
if not Path(uv_executable).is_file():
    raise RuntimeError(f"uv installation failed: {uv_executable}")

packages = [
    "transformers==4.46.3",
    "huggingface_hub>=0.26.0",
    "pyarrow>=16.0.0",
    "remotezip>=0.12.3",
    "requests>=2.31.0",
    "einops>=0.8.0",
    "timm>=1.0.0",
    "Pillow>=10.0.0",
    "numpy>=1.26.0",
    "tqdm>=4.66.0",
    "safetensors>=0.4.0",
    "qwen-vl-utils"
]
subprocess.check_call([uv_executable, "pip", "install", "--system", *packages])
subprocess.check_call([sys.executable, "-m", "pip", "install", "flash-attn", "--no-build-isolation"])

if (PROJECT_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "fetch", "origin", CODE_COMMIT])
elif PROJECT_DIR.exists():
    raise RuntimeError(f"{PROJECT_DIR} exists but is not a git checkout; remove or rename it")
else:
    subprocess.check_call(["git", "clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(PROJECT_DIR)])
    subprocess.check_call(["git", "-C", str(PROJECT_DIR), "fetch", "origin", CODE_COMMIT])
subprocess.check_call(["git", "-C", str(PROJECT_DIR), "checkout", "--detach", CODE_COMMIT])


In [ ]:
import select
import time

def run_streamed(command, *, cwd, env, log_path, heartbeat_seconds=15):
    child_env = env.copy()
    child_env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, command)), flush=True)
    started = time.monotonic()
    last_heartbeat = started
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command, cwd=cwd, env=child_env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=False, bufsize=0,
        )
        assert process.stdout is not None
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 5.0)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if chunk:
                    text_chunk = chunk.decode("utf-8", errors="replace")
                    print(text_chunk, end="", flush=True)
                    log_handle.write(text_chunk)
                    log_handle.flush()
            now = time.monotonic()
            if process.poll() is not None:
                remainder = process.stdout.read()
                if remainder:
                    text_chunk = remainder.decode("utf-8", errors="replace")
                    print(text_chunk, end="", flush=True)
                    log_handle.write(text_chunk)
                    log_handle.flush()
                break
            if now - last_heartbeat >= heartbeat_seconds:
                message = f"[still running] elapsed={(now - started) / 60:.1f} min | log={log_path}\n"
                print(message, end="", flush=True)
                log_handle.write(message)
                log_handle.flush()
                last_heartbeat = now
    return process.returncode

OFFICIAL_BUNDLE_DIR = Path("/content/conformal_official_bundle")
prepare_script = PROJECT_DIR / "scripts" / "prepare_official_conformal_bundle.py"
prepare_command = [
    sys.executable, str(prepare_script),
    "--output-dir", str(OFFICIAL_BUNDLE_DIR),
    "--datasets", "mmqa,webqa,hotpotqa,tatqa",
    "--max-questions", "0",
]
prepare_log = Path("/content/official_bundle_prepare.log")
return_code = run_streamed(prepare_command, cwd=PROJECT_DIR, env=os.environ, log_path=prepare_log)
if return_code != 0:
    raise RuntimeError(f"Failed to download bundle. See {prepare_log}")


In [ ]:
import torch
import string
import re
from collections import Counter
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import pandas as pd
from google.colab import files

def normalize_answer(s):
    def remove_articles(text): return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text): return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    def lower(text): return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_f1(a_gold, a_pred):
    gold_toks = normalize_answer(a_gold).split()
    pred_toks = normalize_answer(a_pred).split()
    common = Counter(gold_toks) & Counter(pred_toks)
    num_same = sum(common.values())
    if len(gold_toks) == 0 or len(pred_toks) == 0:
        return int(gold_toks == pred_toks)
    if num_same == 0:
        return 0
    precision = 1.0 * num_same / len(pred_toks)
    recall = 1.0 * num_same / len(gold_toks)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1

def compute_em(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def metric_max_over_ground_truths(metric_fn, prediction, ground_truths):
    scores_for_ground_truths = []
    for ground_truth in ground_truths:
        score = metric_fn(ground_truth, prediction)
        scores_for_ground_truths.append(score)
    return max(scores_for_ground_truths)


In [ ]:
print("Vui lòng upload file downstream_export.json!")
uploaded = files.upload()
export_data = json.load(open("downstream_export.json", encoding="utf-8"))

corpus_map = {}
gold_map = {}
datasets = ["hotpotqa", "mmqa", "tatqa", "webqa"]
for ds in datasets:
    corpus_map[ds] = {}
    with open(OFFICIAL_BUNDLE_DIR / ds / "corpus.jsonl", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            corpus_map[ds][row["id"]] = row
    
    with open(OFFICIAL_BUNDLE_DIR / ds / "questions.jsonl", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            gold_map[row["qid"]] = row["gold_answers"]


In [ ]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct", 
    torch_dtype=torch.bfloat16, 
    device_map="auto",
    attn_implementation="flash_attention_2"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")

methods = list(export_data[0]["methods"].keys())
results = []

for method in methods:
    print(f"\n========== ĐANG CHẠY METHOD: {method} ==========")
    method_ems, method_f1s, method_latencies, method_flops = [], [], [], []
    
    for idx, row in enumerate(export_data):
        ds = row["dataset"]
        qid = row["qid"]
        q_text = row["query_text"]
        chunks = row["methods"][method]
        gold_answers = gold_map.get(qid, [])
        
        content_list = []
        for c in chunks:
            chunk_data = corpus_map[ds].get(c["chunk_id"])
            if not chunk_data: continue
            if chunk_data["modality"] in ["text", "table"]:
                content_list.append({"type": "text", "text": f"Context: {chunk_data['content']}"})
            elif chunk_data["modality"] == "image":
                img_path = OFFICIAL_BUNDLE_DIR / chunk_data['content']
                content_list.append({"type": "image", "image": f"file://{img_path}"})
        
        content_list.append({"type": "text", "text": f"Question: {q_text}\nAnswer:"})
        messages = [{"role": "user", "content": content_list}]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)
        
        start_time = time.time()
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=50)
        latency = time.time() - start_time
        
        num_params = sum(p.numel() for p in model.parameters())
        flops = 2 * num_params * generated_ids.shape[1]
        
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        pred_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        
        em = metric_max_over_ground_truths(compute_em, pred_text, gold_answers)
        f1 = metric_max_over_ground_truths(compute_f1, pred_text, gold_answers)
        
        method_ems.append(em)
        method_f1s.append(f1)
        method_latencies.append(latency)
        method_flops.append(flops)
        
        print(f"[{method}] {idx+1}/{len(export_data)} | {ds} | EM: {em} | F1: {f1:.2f} | Lat: {latency:.2f}s | TFlops: {flops/1e12:.2f}")
        
    avg_em = sum(method_ems) / len(method_ems) * 100
    avg_f1 = sum(method_f1s) / len(method_f1s) * 100
    avg_lat = sum(method_latencies) / len(method_latencies)
    avg_flops = sum(method_flops) / len(method_flops) / 1e12
    
    res = {"Method": method, "EM": avg_em, "F1": avg_f1, "Latency (s)": avg_lat, "TFLOPS": avg_flops}
    results.append(res)
    print(f"--- Hoàn thành {method} | EM: {avg_em:.2f}% | F1: {avg_f1:.2f}% | Latency: {avg_lat:.2f}s | TFLOPS: {avg_flops:.2f} ---")
    
    df = pd.DataFrame(results)
    csv_name = f"results_after_{method}.csv"
    df.to_csv(csv_name, index=False)
    files.download(csv_name)

print("\nĐã chạy xong toàn bộ các method!")
final_csv = "final_downstream_results.csv"
pd.DataFrame(results).to_csv(final_csv, index=False)
files.download(final_csv)
